# 4. Segmentación de clientes (RFM + k-medias)

**Pregunta:** ¿qué tipos de cliente tenemos y qué hacer con cada uno?

**Una fila = un cliente** (sin CLIENTES VARIOS, que es el mostrador sin identificar).

**Método:** RFM, el clásico del marketing, más agrupación automática:
- **R (recencia):** días desde la última compra. Cuanto menos, mejor.
- **F (frecuencia):** compras al año. Se anualiza para no penalizar a los clientes recién llegados.
- **M (monetario):** ventas netas acumuladas.

El algoritmo **k-medias** agrupa a los clientes parecidos en esas tres medidas. No hay variable objetivo: es aprendizaje **no supervisado**.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

from comun import *   # conexion al almacen (con, q), CORTE, guardar(), estilo(), mape(), colores

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

## 1. Tabla RFM

In [ ]:
rfm = q(f'''
    WITH ventas AS (
        SELECT c.cliente_id, CAST(f.fecha AS DATE) AS fecha, v.serie || '-' || v.numero AS documento,
               v.importe_neto_pen, v.margen_pen, td.signo
        FROM fact_ventas v
        JOIN dim_cliente c         ON c.cliente_key = v.cliente_key
        JOIN dim_fecha f           ON f.fecha_key = v.fecha_emision_key
        JOIN dim_tipo_documento td ON td.tipo_documento_key = v.tipo_documento_key
        WHERE c.cliente_id <> '00000000')
    SELECT v.cliente_id, ANY_VALUE(c.nombre) AS cliente, ANY_VALUE(c.tipo_cliente) AS tipo_cliente,
           ANY_VALUE(c.sector) AS sector, ANY_VALUE(c.departamento) AS departamento,
           ANY_VALUE(c.vendedor_asignado) AS vendedor_asignado,
           DATE_DIFF('day', MAX(v.fecha) FILTER (WHERE v.signo = 1), DATE '{CORTE.date()}') AS recencia_dias,
           COUNT(DISTINCT v.documento) FILTER (WHERE v.signo = 1) AS frecuencia,
           ROUND(SUM(v.importe_neto_pen), 2) AS monetario_pen, ROUND(SUM(v.margen_pen), 2) AS margen_pen,
           DATE_DIFF('day', MIN(v.fecha), DATE '{CORTE.date()}') AS dias_desde_primera_compra
    FROM ventas v JOIN dim_cliente c ON c.cliente_id = v.cliente_id AND c.es_actual
    GROUP BY v.cliente_id
''')
rfm = rfm[rfm.monetario_pen > 0].copy()
rfm['compras_al_anio'] = (rfm.frecuencia / np.maximum(rfm.dias_desde_primera_compra, 30) * 365).round(1)
print(len(rfm), 'clientes')
rfm[['recencia_dias', 'compras_al_anio', 'monetario_pen']].describe().round(0)

## 2. Preparación

Las tres medidas están muy sesgadas (pocos clientes compran muchísimo). Se aplica **logaritmo** para que esos pocos no dominen los grupos y luego se **estandarizan** para que las tres pesen lo mismo (si no, el importe, en miles, aplastaría a la frecuencia).

In [ ]:
X = StandardScaler().fit_transform(np.column_stack([
    np.log1p(rfm.recencia_dias), np.log1p(rfm.compras_al_anio), np.log1p(rfm.monetario_pen)]))

## 3. ¿Cuántos grupos? (coeficiente de silueta)

La silueta mide lo separados que quedan los grupos (de −1 a 1; más alto, mejor).

In [ ]:
silueta = {k: round(silhouette_score(X, KMeans(k, n_init=100, random_state=SEMILLA).fit_predict(X)), 3) for k in range(2, 8)}
silueta

**Decisión:** con 2 grupos la silueta es máxima, pero solo separa clientes activos de inactivos, y eso no sirve para actuar. Se eligen **4 grupos**: es el menor número que da segmentos con una acción comercial distinta, y su silueta (≈ 0,33) indica una estructura aceptable. **Esta decisión hay que justificarla así en la memoria:** combina el criterio estadístico con la utilidad para el negocio.

**Estabilidad (importante):** k-medias arranca de centros al azar y, con k = 4, hay varias soluciones casi igual de buenas. Con pocos arranques (`n_init=20`) el resultado puede cambiar de un ordenador a otro: en una primera ejecución salieron 8 clientes clave en lugar de 14. Por eso se usan **100 arranques** (`n_init=100`), con los que el algoritmo llega siempre a la misma solución, la de menor inercia. La celda siguiente lo comprueba con 10 semillas distintas. El grupo de inactivos es estable en todos los casos; lo que se movía era la frontera entre clave, habituales y ocasionales.

In [ ]:
K = 4
km = KMeans(K, n_init=100, random_state=SEMILLA).fit(X)
rfm['grupo'] = km.labels_

# k-medias numera los grupos al azar: se les pone nombre según su perfil
perfil = rfm.groupby('grupo').agg(rec=('recencia_dias', 'median'), mon=('monetario_pen', 'median'), frec=('compras_al_anio', 'median'))
nombres = {perfil.mon.idxmax(): '1 CLAVE'}
nombres[perfil.drop(list(nombres)).rec.idxmax()] = '4 INACTIVOS'
resto = perfil.drop(list(nombres))
nombres[resto.frec.idxmax()] = '2 HABITUALES'
nombres[resto.frec.idxmin()] = '3 OCASIONALES'
rfm['segmento'] = rfm.grupo.map(nombres)

In [ ]:
from sklearn.metrics import adjusted_rand_score

# Se repite el agrupamiento con 10 semillas distintas y se compara con el elegido (1 = idéntico)
estabilidad = [adjusted_rand_score(km.labels_, KMeans(K, n_init=100, random_state=s).fit_predict(X)) for s in range(10)]
print(f'Índice de Rand ajustado frente a 10 semillas: media {np.mean(estabilidad):.3f}, mínimo {np.min(estabilidad):.3f}')
print('Inercia de la solución elegida:', round(km.inertia_, 2))

## 4. Alerta de abandono

Una regla sencilla complementa los grupos: un cliente activo con al menos 3 compras que lleva sin comprar **más del doble de su intervalo habitual** (y más de 60 días) está en riesgo de irse.

In [ ]:
intervalo = rfm.dias_desde_primera_compra / np.maximum(rfm.frecuencia - 1, 1)
rfm['intervalo_habitual_dias'] = intervalo.round(0)
rfm['alerta_abandono'] = np.where((rfm.segmento != '4 INACTIVOS') & (rfm.frecuencia >= 3) &
                                  (rfm.recencia_dias > np.maximum(2 * intervalo, 60)), 'SI', 'NO')
guardar(rfm.drop(columns='grupo').sort_values(['segmento', 'monetario_pen'], ascending=[True, False]), 'segmentos_clientes')

## 5. Perfil de cada segmento

In [ ]:
total = rfm.monetario_pen.sum()
resumen = rfm.groupby('segmento').agg(
    clientes=('cliente_id', 'size'), ventas_pen=('monetario_pen', 'sum'), margen_pen=('margen_pen', 'sum'),
    recencia_mediana_dias=('recencia_dias', 'median'), compras_al_anio_mediana=('compras_al_anio', 'median'),
    ventas_mediana_por_cliente_pen=('monetario_pen', 'median'),
    en_alerta_abandono=('alerta_abandono', lambda s: int((s == 'SI').sum()))).reset_index()
resumen['pct_clientes'] = (resumen.clientes / resumen.clientes.sum() * 100).round(1)
resumen['pct_ventas'] = (resumen.ventas_pen / total * 100).round(1)
resumen['margen_pct'] = (resumen.margen_pen / resumen.ventas_pen * 100).round(1)
resumen['accion_propuesta'] = resumen.segmento.map({
    '1 CLAVE': 'Cuidar: trato personal, condiciones preferentes, no perderlos',
    '2 HABITUALES': 'Crecer: venta cruzada (ver reglas de asociacion) para subir su ticket',
    '3 OCASIONALES': 'Activar: seguimiento comercial y ofertas por proyecto',
    '4 INACTIVOS': 'Recuperar o descartar: campana de reactivacion de bajo coste'})
resumen = resumen.round(1)
guardar(resumen, 'segmentos_resumen')
resumen

In [ ]:
colores = {'1 CLAVE': AZUL, '2 HABITUALES': VERDE, '3 OCASIONALES': NARANJA, '4 INACTIVOS': GRIS}
fig, ax = plt.subplots(figsize=(8.5, 4.6))
for s, g in rfm.groupby('segmento'):
    ax.scatter(g.recencia_dias, g.monetario_pen / 1000, s=12 + g.compras_al_anio * 2, alpha=.7, color=colores[s],
               label=f"{s.split(' ', 1)[1].capitalize()} ({len(g)})", edgecolor='white', lw=.5)
ax.set_xscale('symlog', linthresh=10); ax.set_yscale('log')
ax.set_xlabel('Días desde la última compra'); ax.set_ylabel('Ventas acumuladas (miles de S/, escala log)')
estilo(ax, 'Segmentos de clientes (tamaño = compras al año)')
ax.legend(frameon=False, fontsize=8)
guardar_figura(fig, '4_segmentos')
plt.show()

## Conclusiones (para la memoria)

- Unos **14 clientes clave** (≈ 8 %) hacen más de la mitad de las ventas: confirma la concentración del indicador VEN-06 y es el principal riesgo comercial.
- Un cuarto de los clientes está **inactivo** (más de un año sin comprar) y apenas pesa en ventas.
- Los **ocasionales** son el grupo más numeroso y donde están las alertas de abandono: es donde el seguimiento comercial tiene más recorrido.
- Los segmentos son descriptivos y deben revisarse periódicamente (cada trimestre).